Separate code for training & evaluating / selecting HMM models (i.e. after preparation of dataset from time-series data).

Code will read the 'dataset_selector' and related YAML fields to automatically or manually select the input FROZEN split dataset.

Subsetting / filtering is no longer active as of this stage, as the input training & evaluation data will have been fully prepared at this point.

[Runtime: Entirely dependent on number and complexity of models (protocols) being run; but in most cases should be well under an hour.]

---------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import os, subprocess
import json
import pandas as pd
import numpy as np
import re
from sklearn.decomposition import PCA
from hmmlearn.hmm import GaussianHMM
import joblib

In [ ]:
# __________________________________________________________________________________________________________
### LOAD PARAMETERS:

# General parameters:
HARD_STOP    = config['hard_errors']
RANDOM_SEED  = config['random_seed']

OVERWRITE_MODELS    = bool(config["HMM_training"]["overwrite_models"])

# Training parameters:

HMM_ENABLED = config['HMM_training']['enabled']
PROTOCOLS_TO_RUN    = list(config["HMM_training"]["protocols_to_run"])

HMM_DEFAULTS = config["HMM_protocols"]["defaults"]
HMM_PRESETS  = config["HMM_protocols"]["presets"]

# Evaluation parameters:

AUTOMATIC_SELECTION = config['HMM_evaluation']['automatic_selection']
SELECTION_CRITERION = config['HMM_evaluation']['criteria']
REFIT_ON_ALL_DATA   = config['HMM_evaluation']['refit_on_all_data']


# __________________________________________________________________________________________________________
### SET FILEPATHS:

BASE_DIRECTORY = Path(config['root_output_directory'])

RUN_MANIFEST_PATH    = BASE_DIRECTORY / 'subject_manifest.csv'
fMRI_PARAMETERS_PATH = BASE_DIRECTORY / 'fMRI_manifest.csv'


### INPUTS:

# Grab dataset-selection config variables:
DATASET_SELECTOR    = str(config["ML_training"]["dataset_selector"]).strip().lower()
DATASET_MANUAL_PATH = config["ML_training"].get("dataset_path", None)

# Root directory where frozen datasets live (and where LATEST_DATASET.json pointer is stored):
TRAINING_DATA_ROOT = Path(BASE_DIRECTORY) / config["ML_prep"]["training_data_dir"]

# [[[See next cell for actual input routing]]]


### OUTPUTS:
HMM_MODEL_DIRECTORY = config["HMM_training"]["HMM_model_directory"]
OUTPUT_DIR = Path(BASE_DIRECTORY) / HMM_MODEL_DIRECTORY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# __________________________________________________________________________________________________________
### INITIALIZATION:

RUN_MANIFEST = pd.read_csv(RUN_MANIFEST_PATH)
fMRI_runs    = pd.read_csv(fMRI_PARAMETERS_PATH)

Use config.yaml fields to retrieve target dataset (uses 'dataset_selector' and possibly 'dataset_path' fields, depending on user setup):

In [ ]:
if DATASET_SELECTOR == "latest":
    pointer_path = TRAINING_DATA_ROOT / "LATEST_DATASET.json"
    if not pointer_path.exists():
        raise FileNotFoundError(
            f"[INIT ERROR] dataset_selector='latest' but pointer file not found:\n  {pointer_path}")
    with open(pointer_path, "r") as f:
        pointer = json.load(f)
    DATASET_DIR = Path(pointer["dataset_dir"])

elif DATASET_SELECTOR == "manual":
    if DATASET_MANUAL_PATH is None or str(DATASET_MANUAL_PATH).strip() == "":
        raise ValueError(
            "[INIT ERROR] dataset_selector='manual' but ML_training.dataset_path is empty.")
    # User supplies a full path to the frozen dataset directory:
    DATASET_DIR = Path(str(DATASET_MANUAL_PATH)).expanduser()

else:
    raise ValueError(
        f"[INIT ERROR] ML_training.dataset_selector must be 'latest' or 'manual' "
        f"(got: {DATASET_SELECTOR})")

# Set paths to expected input files:
x_all_path       = DATASET_DIR / "X_all.csv"
x_train_path     = DATASET_DIR / "X_train.csv"
x_test_path      = DATASET_DIR / "X_test.csv"
provenance_path  = DATASET_DIR / "provenance.json"

# Sanity check: verify that all required files actually exist (note: X_test.csv may be *empty*, but it must still exist):
required_files = {
    "X_all": x_all_path,
    "X_train": x_train_path,
    "X_test": x_test_path,
    "provenance": provenance_path}
missing_files = {name: path for name, path in required_files.items() if not path.exists()}
if missing_files:
    missing_str = "\n".join(f"  - {name}: {path}" for name, path in missing_files.items())
    raise FileNotFoundError(
        "[INIT ERROR] Frozen dataset directory is missing required file(s):\n"
        f"{missing_str}")

# Report resolved dataset:
print("\n[INIT] Frozen dataset target resolved:")
print(f"   TRAINING_DATA_ROOT       = {TRAINING_DATA_ROOT}")
print(f"   dataset selection mode   = '{DATASET_SELECTOR}'")
print(f"   DATASET_DIR              = {DATASET_DIR}")
print(f"     X_all                  = {x_all_path}")
print(f"     X_train                = {x_train_path}")
print(f"     X_test                 = {x_test_path}")
print(f"     provenance             = {provenance_path}")

Quick "pre-flight" check of chosen analysis parameters (i.e. the chosen protocol presets):

In [ ]:
# __________________________________________________________________________________________________________
### HMM PROTOCOL PRE-FLIGHT: validate presets + print effective settings + warnings
#
# Uses:
#   - PROTOCOLS_TO_RUN, HMM_DEFAULTS, HMM_PRESETS
#   - DATASET_DIR, OUTPUT_DIR, OVERWRITE_MODELS
#   - provenance.json (optional) to warn if dataset was not standardized/centered
#
# Outputs (in-memory):
#   - EFFECTIVE_PROTOCOLS : dict[protocol_name -> dict of resolved hyperparameters]
#     (used downstream by training loop)
# __________________________________________________________________________________________________________

# Optional: inspect provenance for scaling context (for PCA advisories):
provenance_scaling = None
try:
    if provenance_path.exists():
        with open(provenance_path, "r") as f:
            _prov = json.load(f)
        if isinstance(_prov, dict) and "scaling" in _prov:
            provenance_scaling = _prov["scaling"]
except Exception:
    provenance_scaling = None  # <-- soft-fail; only used for warnings

# Normalize protocol names:
protocols_to_run_norm = []
for p in PROTOCOLS_TO_RUN:
    if p is None:
        continue
    p_str = str(p).strip()
    if p_str:
        protocols_to_run_norm.append(p_str)

if not protocols_to_run_norm:
    raise RuntimeError("[PRE-FLIGHT ERROR] HMM_training.protocols_to_run is empty; nothing to run.")

# Validate presets exist:
missing_presets = [p for p in protocols_to_run_norm if p not in HMM_PRESETS]
if missing_presets:
    raise KeyError(
        "[PRE-FLIGHT ERROR] The following protocol(s) were requested in "
        "HMM_training.protocols_to_run but are missing from HMM_protocols.presets:\n"
        + "\n".join(f"  - {p}" for p in missing_presets))

# Helper: default fallback logic (treat None/'null'/'' as "use defaults"):
def _use_default(val):
    if val is None:
        return True
    if isinstance(val, str) and val.strip().lower() in {"null", "none", ""}:
        return True
    return False

# Resolve effective protocol configs:
EFFECTIVE_PROTOCOLS = {}

print("\n[PRE-FLIGHT] HMM protocols selected:")
print("  " + ", ".join(protocols_to_run_norm))

# Dataset-scoped output folder name:
DATASET_OUTPUT_ROOT = OUTPUT_DIR / DATASET_DIR.name

if DATASET_OUTPUT_ROOT.exists() and (not OVERWRITE_MODELS):
    print(
        f"[PRE-FLIGHT WARN] overwrite_models=False and dataset output directory already exists:\n"
        f"  {DATASET_OUTPUT_ROOT}\n"
        f"  Training may fail later when trying to write outputs.")

for proto_name in protocols_to_run_norm:
    preset = HMM_PRESETS[proto_name]

    # Required-ish fields:
    cov_type = str(preset.get("covariance_type", "")).strip().lower()
    pca_block = preset.get("PCA", {}) if isinstance(preset.get("PCA", {}), dict) else {}

    pca_enabled = bool(pca_block.get("enabled", False))
    pca_n_components = pca_block.get("n_components", None)
    pca_whitening = bool(pca_block.get("whitening", False))

    # Default fallback for core HMM settings:
    k_values = preset.get("k_values", None)
    if _use_default(k_values):
        k_values = HMM_DEFAULTS["k_values"]

    n_init = preset.get("num_initializations", None)
    if _use_default(n_init):
        n_init = HMM_DEFAULTS["num_initializations"]

    max_iter = preset.get("max_iterations", None)
    if _use_default(max_iter):
        max_iter = HMM_DEFAULTS["max_iterations"]

    tol = preset.get("convergence_tolerance", None)
    if _use_default(tol):
        tol = HMM_DEFAULTS["convergence_tolerance"]

    # Store effective params:
    EFFECTIVE_PROTOCOLS[proto_name] = {
        "covariance_type": cov_type,
        "PCA": {
            "enabled": pca_enabled,
            "n_components": pca_n_components,
            "whitening": pca_whitening},
        "k_values": list(k_values),
        "num_initializations": int(n_init),
        "max_iterations": int(max_iter),
        "convergence_tolerance": float(tol),
        "dataset_output_dir": str(DATASET_OUTPUT_ROOT / proto_name)}

    # Print summary:
    print(f"\n[PRE-FLIGHT] Protocol '{proto_name}' (effective):")
    print(f"  covariance_type        = '{cov_type}'")
    print(f"  PCA.enabled            = {pca_enabled}")
    print(f"  PCA.n_components       = {pca_n_components}")
    print(f"  PCA.whitening          = {pca_whitening}")
    print(f"  k_values               = {list(k_values)}")
    print(f"  num_initializations    = {int(n_init)}")
    print(f"  max_iterations         = {int(max_iter)}")
    print(f"  convergence_tolerance  = {float(tol)}")
    print(f"  output_dir             = {DATASET_OUTPUT_ROOT / proto_name}")

    # ---- Warnings (non-fatal) ----

    # 1) Full covariance without PCA:
    if cov_type == "full" and not pca_enabled:
        print(
            "[PRE-FLIGHT WARN] covariance_type='full' with PCA disabled. "
            "With many features this can be slow/unstable; consider enabling PCA or using 'diag'.")

    # 2) PCA enabled but dataset scaling context suggests it may not be standardized/centered:
    if pca_enabled and isinstance(provenance_scaling, dict):
        centered = bool(provenance_scaling.get("centered", False))
        standardized = bool(provenance_scaling.get("standardized", False))
        if not centered:
            print(
                "[PRE-FLIGHT WARN] PCA is enabled, but provenance suggests the frozen dataset was not mean-centered. "
                "PCA generally assumes centering at minimum.")
        if not standardized:
            print(
                "[PRE-FLIGHT WARN] PCA is enabled, but provenance suggests the frozen dataset was not standardized. "
                "If features are on different scales, PCA can be dominated by large-variance features.")

    # 3) PCA whitening with diag covariance (fine, but note interpretation):
    if pca_enabled and pca_whitening and cov_type == "diag":
        print(
            "[PRE-FLIGHT NOTE] PCA whitening + diagonal covariance is valid, but note that whitening already sets "
            "PC variances ~1; diagonal covariance then re-estimates per-state variances in PC space.")

    # 4) Quick sanity-check on k_values:
    if not isinstance(k_values, (list, tuple)) or len(k_values) == 0:
        print("[PRE-FLIGHT WARN] k_values is empty; this protocol will train no models.")
    else:
        bad_k = [k for k in k_values if (not isinstance(k, (int, np.integer))) or int(k) < 2]
        if bad_k:
            print(f"[PRE-FLIGHT WARN] Some k_values are invalid (must be int >= 2): {bad_k}")

print("\n[PRE-FLIGHT] Protocol vetting complete.")

---------

**Main training loop:**

Step #1: Convert X_train, X_test, and X_all into HMM-legible 2D numpy arrays:

In [ ]:
# __________________________________________________________________________________________________________
### BUILD HMM-READY ARRAYS (+ OPTIONAL TRAIN / VALIDATION SPLIT)
#
# Uses:
#   - Frozen dataset CSVs in DATASET_DIR:
#       X_all.csv, X_train.csv, X_test.csv
#   - provenance.json (optional) for feature column ordering
#
# Behaviour (updated for frozen-dataset workflow):
#   - Reconstructs sequence structure by grouping rows by (subject_ID, session_ID),
#     preserving a stable ordering for concatenation.
#   - Builds:
#       * X_all, lengths_all
#       * X_train, lengths_train
#       * X_validation, lengths_validation  (from X_test; may be empty)
#
# Outputs (in-memory):
#   - feature_columns_final             : list of feature column names (in order)
#   - sequences_all_df / sequences_train_df / sequences_validation_df
#   - X_all, lengths_all
#   - X_train, lengths_train
#   - X_validation, lengths_validation
# __________________________________________________________________________________________________________

def handle_error(message: str):
    if HARD_STOP:
        raise RuntimeError(message)
    else:
        print(f"[WARN] {message}")

# Load frozen tables:
try:
    X_all_df   = pd.read_csv(x_all_path)
    X_train_df = pd.read_csv(x_train_path)
    X_test_df  = pd.read_csv(x_test_path)
except Exception as exc:
    raise RuntimeError(f"[HMM PREP ERROR] Failed to read frozen dataset CSVs: {exc}")

# Basic sanity-checks for required ID columns:
id_columns = ["subject_ID", "session_ID", "time_index"]
for df_name, df in [("X_all_df", X_all_df), ("X_train_df", X_train_df), ("X_test_df", X_test_df)]:
    missing = [c for c in id_columns if c not in df.columns]
    if missing:
        handle_error(f"{df_name} is missing required ID column(s): {missing}")

# Determine feature column order (prefer provenance-file data, if present):
feature_columns_final = None
provenance = None

if provenance_path.exists():
    try:
        with open(provenance_path, "r") as f:
            provenance = json.load(f)
        if isinstance(provenance, dict) and "feature_columns" in provenance:
            feature_columns_final = list(provenance["feature_columns"])
    except Exception as exc:
        handle_error(f"Could not read provenance.json for feature order; will infer from CSV columns. Error: {exc}")

if feature_columns_final is None:
    # Infer from 'X_train' (preferred) else 'X_all':
    candidate_df = X_train_df if not X_train_df.empty else X_all_df
    feature_columns_final = [c for c in candidate_df.columns if c not in id_columns]
    feature_columns_final = sorted(feature_columns_final)

# Verify feature columns exist in all tables (e.g. 'X_test' may be empty but should still have header):
for df_name, df in [("X_all_df", X_all_df), ("X_train_df", X_train_df), ("X_test_df", X_test_df)]:
    missing_feats = [c for c in feature_columns_final if c not in df.columns]
    if missing_feats:
        handle_error(f"{df_name} is missing feature column(s) required by feature_columns_final: {missing_feats[:10]}")

if not feature_columns_final:
    handle_error("No feature columns found; only ID columns present.")

# Helper function: build (X, lengths, sequences_df) from table:
def build_X_and_lengths_from_table(df: pd.DataFrame, feature_cols: list) -> tuple:
    """
    Reconstruct sequence descriptors by grouping by (subject_ID, session_ID).
    Returns:
      X (np.ndarray), lengths (list[int]), sequences_df (pd.DataFrame)
    """
    if df.empty:
        X = np.empty((0, len(feature_cols)), dtype=float)
        lengths = []
        sequences_df = pd.DataFrame(columns=["subject_ID", "session_ID", "length", "row_start", "row_end"])
        return X, lengths, sequences_df

    # Ensure stable ordering within dataframe first:
    df_sorted = df.sort_values(["subject_ID", "session_ID", "time_index"]).reset_index(drop=True)

    # Build sequences in the order they appear ('groupby sort=False' preserves first-seen order):
    seq_rows = []
    row_cursor = 0

    for (subj, sess), g in df_sorted.groupby(["subject_ID", "session_ID"], sort=False):
        L = int(g.shape[0])
        if L <= 0:
            continue
        seq_rows.append({
            "subject_ID": str(subj),
            "session_ID": str(sess),
            "length": L,
            "row_start": int(row_cursor),
            "row_end": int(row_cursor + L - 1)})
        row_cursor += L

    sequences_df = pd.DataFrame(seq_rows)
    if sequences_df.empty:
        X = np.empty((0, len(feature_cols)), dtype=float)
        lengths = []
        return X, lengths, sequences_df

    # Build the numeric matrix aligned to the same ordering used above:
    X = df_sorted[feature_cols].to_numpy(dtype=float)
    lengths = sequences_df["length"].astype(int).tolist()
    return X, lengths, sequences_df

# Build arrays:
X_all, lengths_all, sequences_all_df = build_X_and_lengths_from_table(X_all_df, feature_columns_final)
X_train, lengths_train, train_sequences_df = build_X_and_lengths_from_table(X_train_df, feature_columns_final)

# Use 'X_test' as the validation set in the training script:
X_validation, lengths_validation, validation_sequences_df = build_X_and_lengths_from_table(X_test_df, feature_columns_final)

print("\n[HMM PREP] Built feature matrices from frozen dataset:")
print(f"   X_all shape         = {X_all.shape}   (# sequences: {len(lengths_all)})")
print(f"   X_train shape       = {X_train.shape}   (# sequences: {len(lengths_train)})")
print(f"   X_validation shape  = {X_validation.shape}   (# sequences: {len(lengths_validation)})")

if lengths_all:
    print(f"   lengths_all         = (min={min(lengths_all)}, max={max(lengths_all)})")
if lengths_train:
    print(f"   lengths_train       = (min={min(lengths_train)}, max={max(lengths_train)})")
if lengths_validation:
    print(f"   lengths_validation  = (min={min(lengths_validation)}, max={max(lengths_validation)})")

### Derived flag for downstream scoring logic:
# Validate that 'X_test' exists and is not empty (gates downstream logic appropriately):
HAS_TEST_SET = (
    X_validation is not None and
    X_validation.shape[0] > 0 and
    lengths_validation is not None and
    len(lengths_validation) > 0 and
    int(np.sum(lengths_validation)) == int(X_validation.shape[0]))
if not HAS_TEST_SET:
    print("[HMM PREP] NOTE: Validation set (X_test) is empty --> val_logL will be skipped (NaN).")

Next, perform model training, across all user-selected protocols:

In [ ]:
# __________________________________________________________________________________________________________
### HMM TRAINING (per protocol, per K_value, per init):
#
# Outputs (in-memory):
#   - PROTOCOL_TRAINED_MODELS        : dict[protocol][K] -> dict(model + metrics + preprocessing artifacts)
#   - PROTOCOL_COMPARISON_DF         : dict[protocol] -> DataFrame (one row per K)
#
# Side effects (on disk):
#   - Saves per-protocol comparison CSV/JSON in:
#       OUTPUT_DIR / <DATASET_DIR.name> / <protocol_name> /
#   - Saves best-per-K model objects via joblib in same folder
#   - Saves PCA object (if used) for that protocol in same folder
# __________________________________________________________________________________________________________

if not HMM_ENABLED:
    print("[HMM] HMM modeling is disabled (HMM_ENABLED=False). Skipping training.")
else:
    # Helper: robust random_state handling:
    seed_is_int = isinstance(RANDOM_SEED, (int, np.integer))
    base_seed = int(RANDOM_SEED) if seed_is_int else None

    # Initialize storage:
    PROTOCOL_TRAINED_MODELS = {}
    PROTOCOL_COMPARISON_DF = {}

    # Train each protocol independently:
    for protocol_name, proto_cfg in EFFECTIVE_PROTOCOLS.items():
        print("\n" + "=" * 88)
        print(f"[HMM] Protocol: {protocol_name}")
        print("=" * 88)

        # Resolve per-protocol settings:
        cov_type = str(proto_cfg["covariance_type"]).strip().lower()
        k_values = list(proto_cfg["k_values"])
        n_init = int(proto_cfg["num_initializations"])
        max_iter = int(proto_cfg["max_iterations"])
        tol = float(proto_cfg["convergence_tolerance"])

        pca_enabled = bool(proto_cfg["PCA"]["enabled"])
        pca_n_components = proto_cfg["PCA"]["n_components"]
        pca_whitening = bool(proto_cfg["PCA"]["whitening"])

        # Set output directory for this protocol under the main dataset folder:
        protocol_dir = Path(proto_cfg["dataset_output_dir"])
        protocol_dir.mkdir(parents=True, exist_ok=True)

        # Optional PCA: fit on 'X_train' ONLY, then transform train / all / validation:
        pca_obj = None
        X_train_eff = X_train
        X_all_eff = X_all
        X_val_eff = X_validation

        pca_info = {
            "enabled": pca_enabled,
            "n_components_requested": pca_n_components,
            "n_components_effective": None,
            "whitening": pca_whitening,
            "explained_variance_ratio": None,
            "explained_variance_cumulative": None}

        if pca_enabled:
            print("[HMM] PCA enabled: fitting PCA on X_train, applying to X_train/X_all/X_validation.")
            pca_obj = PCA(
                n_components=pca_n_components,
                whiten=pca_whitening,
                random_state=base_seed)
            X_train_eff = pca_obj.fit_transform(X_train)
            X_all_eff = pca_obj.transform(X_all)

            if HAS_TEST_SET:
                X_val_eff = pca_obj.transform(X_validation)
            else:
                X_val_eff = X_validation

            evr = getattr(pca_obj, "explained_variance_ratio_", None)
            if evr is not None:
                pca_info["n_components_effective"] = int(X_train_eff.shape[1])
                pca_info["explained_variance_ratio"] = evr.tolist()
                pca_info["explained_variance_cumulative"] = float(np.cumsum(evr)[-1])

            print(f"[HMM] PCA: features {X_train.shape[1]} -> {X_train_eff.shape[1]}")
            if pca_info["explained_variance_cumulative"] is not None:
                print(f"[HMM] PCA: cumulative explained variance = {pca_info['explained_variance_cumulative']:.3f}")

            # Save PCA object + brief JSON sidecar:
            joblib.dump(pca_obj, protocol_dir / "pca_fit_on_X_train.joblib")
            with open(protocol_dir / "pca_fit_on_X_train.json", "w") as f:
                json.dump(pca_info, f, indent=2)

        # Bookkeeping:
        n_train_obs, n_features = X_train_eff.shape
        print(f"[HMM] K candidates: {k_values}")
        print(f"[HMM] covariance_type={cov_type}, n_init={n_init}, max_iter={max_iter}, tol={tol}")
        print(f"[HMM] Training observations: {n_train_obs}, features: {n_features}")
        print(f"[HMM] Validation scoring: {'enabled' if HAS_TEST_SET else 'skipped (no test set)'}")
        print(f"[HMM] Output dir: {protocol_dir}")
        trained_hmm_models = {}
        comparison_rows = []

        for K in k_values:
            print(f"\n[HMM] ----- K={K} -----")

            best_model_for_K = None
            best_init_index = None
            best_train_logL = -np.inf
            best_val_logL = np.nan

            # Calculate approximate total parameter count in the *effective* feature space:
            D = int(n_features)
            if cov_type == "full":
                n_cov_params = K * D * (D + 1) / 2.0
            else:
                n_cov_params = K * D
            n_mean_params = K * D
            n_trans_params = K * (K - 1)
            n_startprob_params = K - 1
            num_params = int(n_mean_params + n_cov_params + n_trans_params + n_startprob_params)

            for init_idx in range(n_init):
                # Reproducible diversity if 'RANDOM_SEED' is an int; else use None:
                random_state = (base_seed + init_idx) if seed_is_int else None

                try:
                    model = GaussianHMM(
                        n_components=int(K),
                        covariance_type=cov_type,
                        n_iter=max_iter,
                        tol=tol,
                        random_state=random_state,
                        verbose=False)
                    model.fit(X_train_eff, lengths_train)
                    train_logL = float(model.score(X_train_eff, lengths_train))
                except Exception as e:
                    print(f"[HMM WARN] K={K}, init={init_idx}: training failed: {e}")
                    continue

                print(f"[HMM] K={K}, init={init_idx}: train logL={train_logL:.3f}")

                if HAS_TEST_SET:
                    try:
                        val_logL = float(model.score(X_val_eff, lengths_validation))
                    except Exception as e:
                        print(f"[HMM WARN] K={K}, init={init_idx}: validation scoring failed: {e}")
                        val_logL = np.nan
                else:
                    val_logL = np.nan

                if train_logL > best_train_logL:
                    best_train_logL = train_logL
                    best_val_logL = val_logL
                    best_model_for_K = model
                    best_init_index = init_idx

            if best_model_for_K is None:
                print(f"[HMM WARN] No successful model trained for K={K}. Skipping.")
                continue

            # AIC/BIC computed on training data
            AIC = 2 * num_params - 2 * best_train_logL
            BIC = num_params * np.log(n_train_obs) - 2 * best_train_logL

            print(f"[HMM] K={K}: best init={best_init_index}, train_logL={best_train_logL:.3f}, AIC={AIC:.2f}, BIC={BIC:.2f}")

            # Save best-per-K model object:
            model_path = protocol_dir / f"HMM_K-{int(K)}.joblib"
            joblib.dump(best_model_for_K, model_path)

            trained_hmm_models[int(K)] = {
                "model": best_model_for_K,
                "model_path": str(model_path),
                "best_init_index": int(best_init_index),
                "num_parameters": int(num_params),
                "num_train_observations": int(n_train_obs),
                "train_logL": float(best_train_logL),
                "AIC": float(AIC),
                "BIC": float(BIC),
                "val_logL": float(best_val_logL) if not np.isnan(best_val_logL) else np.nan,
                "num_val_observations": int(np.sum(lengths_validation)) if HAS_TEST_SET else 0}

            comparison_rows.append({
                "protocol": protocol_name,
                "K": int(K),
                "covariance_type": cov_type,
                "pca_enabled": bool(pca_enabled),
                "pca_n_components": pca_n_components,
                "pca_whitening": bool(pca_whitening),
                "num_parameters": int(num_params),
                "num_train_observations": int(n_train_obs),
                "train_logL": float(best_train_logL),
                "AIC": float(AIC),
                "BIC": float(BIC),
                "val_logL": float(best_val_logL) if not np.isnan(best_val_logL) else np.nan,
                "num_val_observations": int(np.sum(lengths_validation)) if HAS_TEST_SET else 0,
                "best_initialization_index": int(best_init_index),
                "best_model_path": str(model_path)})

        if not comparison_rows:
            raise RuntimeError(f"[HMM ERROR] Protocol '{protocol_name}': no valid HMM models were trained for any K.")

        model_comparison_df = pd.DataFrame(comparison_rows).sort_values("K").reset_index(drop=True)

        print("\n[HMM] Model comparison summary (this protocol):")
        display(model_comparison_df)

        # Save comparisons:
        summary_csv_path  = protocol_dir / "model_comparison.csv"
        summary_json_path = protocol_dir / "model_comparison.json"

        model_comparison_df.to_csv(summary_csv_path, index=False)
        with open(summary_json_path, "w") as f:
            json.dump(model_comparison_df.to_dict(orient="records"), f, indent=2)

        print("\n[HMM] Saved protocol outputs:")
        print(f"  - {summary_csv_path}")
        print(f"  - {summary_json_path}")

        PROTOCOL_TRAINED_MODELS[protocol_name] = {
            "protocol_config": proto_cfg,
            "pca_info": pca_info,
            "pca_object_path": str(protocol_dir / "pca_fit_on_X_train.joblib") if pca_enabled else None,
            "trained_models": trained_hmm_models,
            "comparison_csv": str(summary_csv_path),
            "comparison_json": str(summary_json_path)}
        PROTOCOL_COMPARISON_DF[protocol_name] = model_comparison_df

    print("\n[HMM] Training complete for all requested protocols.")

--------

### OPTIONAL CONTROL CELL: Manual within-protocol model selection:

In [ ]:
# __________________________________________________________________________________________________________
### OPTIONAL: MANUAL OVERRIDES FOR WITHIN-PROTOCOL SELECTION
#
# If AUTOMATIC_SELECTION=True:
#   - You may override any subset of protocols; others will be auto-selected.
#
# If AUTOMATIC_SELECTION=False:
#   - You MUST provide an override for EVERY protocol that was run (and has candidates),
#     otherwise selection cannot proceed and the next cell will hard error.
# __________________________________________________________________________________________________________

CHOSEN_K_BY_PROTOCOL = {
    # Example:
    # "diag_raw": 6,
    # "full_pca90": 7,
}

--------

Next, we compare all K models within each protocol type, and automatically pick the best one for each protocol (for any model families for which no manual overrides were otherwise provided):

In [ ]:
# __________________________________________________________________________________________________________
### WITHIN-PROTOCOL EVALUATION + MODEL SELECTION (+ optional refit on X_all)
#
# Uses:
#   - PROTOCOL_COMPARISON_DF : dict[protocol] -> DataFrame (one row per K; includes metrics + best_model_path)
#   - EFFECTIVE_PROTOCOLS    : dict[protocol] -> resolved protocol params (includes dataset_output_dir, PCA, etc.)
#   - HAS_TEST_SET           : gates val_logL usage
#   - AUTOMATIC_SELECTION, SELECTION_CRITERION, REFIT_ON_ALL_DATA
#   - X_all, lengths_all     : for optional refit
#   - CHOSEN_K_BY_PROTOCOL   : optional overrides (from prior cell)
#
# Outputs (in-memory):
#   - CHOSEN_MODELS_BY_PROTOCOL : dict[protocol] -> dict with chosen K/model/info/paths
#   - winners_df               : compact table (one row per protocol)
#
# Side effects (on disk) per protocol:
#   - chosen_model.json
#   - if REFIT_ON_ALL_DATA=True:
#       * final-model_k{K}.joblib
#       * pca_refit_on_X_all.joblib  (only if PCA was enabled for that specific protocol)
# __________________________________________________________________________________________________________

# Basic sanity-checks:
if "PROTOCOL_COMPARISON_DF" not in globals() or not PROTOCOL_COMPARISON_DF:
    raise RuntimeError("[EVAL ERROR] No protocol comparison tables found. Run the training cell first.")

valid_criteria = {"AIC", "BIC", "val_logL"}
crit_requested = str(SELECTION_CRITERION).strip()
if AUTOMATIC_SELECTION and crit_requested not in valid_criteria:
    raise RuntimeError(
        f"[EVAL ERROR] HMM_evaluation.criteria='{crit_requested}' is invalid. "
        f"Valid options are: {sorted(valid_criteria)}")

# Identify protocols that actually have candidates:
protocols_with_candidates = [
    proto for proto, df in PROTOCOL_COMPARISON_DF.items()
    if df is not None and not df.empty]

if not protocols_with_candidates:
    raise RuntimeError("[EVAL ERROR] All protocol comparison tables are empty; cannot select models.")

# Enforce 'selection_mode' rules:
if not AUTOMATIC_SELECTION:
    missing = [p for p in protocols_with_candidates if p not in CHOSEN_K_BY_PROTOCOL]
    if missing:
        raise RuntimeError(
            "[EVAL ERROR] AUTOMATIC_SELECTION=False, so you must provide CHOSEN_K_BY_PROTOCOL entries "
            f"for every protocol. Missing: {missing}")

# Effective criterion for auto-selection (w/ a global fallback when val_logL is unavailable):
criterion_effective_global = crit_requested
if criterion_effective_global == "val_logL" and not HAS_TEST_SET:
    print("[EVAL WARN] criteria='val_logL' but HAS_TEST_SET=False (X_test empty) --> falling back to 'BIC' within each protocol.")
    criterion_effective_global = "BIC"

def pick_best_row_with_fallback(df: pd.DataFrame, criterion: str) -> dict:
    """
    Returns a row-dict for the best K within THIS df using criterion:
      - AIC/BIC: minimize
      - val_logL: maximize (falls back to train_logL if all NaN)
    """
    df_local = df.copy()
    df_local["K"] = df_local["K"].astype(int)

    if criterion in ("AIC", "BIC"):
        metric = df_local[criterion].astype(float)
        idx = metric.idxmin()
        return df_local.loc[idx].to_dict()

    # val_logL path:
    s = df_local["val_logL"].astype(float)
    non_nan = s.notna()
    if int(non_nan.sum()) == 0:
        metric = df_local["train_logL"].astype(float)
        idx = metric.idxmax()
        return df_local.loc[idx].to_dict()

    idx = s[non_nan].idxmax()
    return df_local.loc[idx].to_dict()

# Iterate over all protocols & choose the best model within each:
CHOSEN_MODELS_BY_PROTOCOL = {}
winner_rows = []

for protocol_name in protocols_with_candidates:
    df = PROTOCOL_COMPARISON_DF[protocol_name].copy()
    df["K"] = df["K"].astype(int)

    proto_cfg = EFFECTIVE_PROTOCOLS[protocol_name]
    protocol_dir = Path(proto_cfg["dataset_output_dir"])
    protocol_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "-" * 88)
    print(f"[EVAL] Protocol: {protocol_name}")
    print("-" * 88)

    # PCA provenance for the *interim* models (trained/scored in training cell):
    pca_cfg = proto_cfg.get("PCA", {}) or {}
    pca_enabled = bool(pca_cfg.get("enabled", False))
    pca_fit_on_train_path = protocol_dir / "pca_fit_on_X_train.joblib"
    pca_fit_on_train_path = str(pca_fit_on_train_path) if (pca_enabled and Path(pca_fit_on_train_path).exists()) else None

    ### Decide chosen K (whether from manual override or auto):
    chosen_by_manual = protocol_name in CHOSEN_K_BY_PROTOCOL
    if chosen_by_manual:
        chosen_k = int(CHOSEN_K_BY_PROTOCOL[protocol_name])
        available_ks = sorted(set(df["K"].tolist()))
        if chosen_k not in set(available_ks):
            raise RuntimeError(
                f"[EVAL ERROR] Manual override CHOSEN_K_BY_PROTOCOL['{protocol_name}']={chosen_k} is not available. "
                f"Available Ks: {available_ks}")
        chosen_row = df.loc[df["K"] == chosen_k].iloc[0].to_dict()
        criterion_used = None
        metric_col_used = None
        print(f"[EVAL] Manual override: chosen K={chosen_k}")
    else:
        if not AUTOMATIC_SELECTION:
            raise RuntimeError(
                f"[EVAL ERROR] AUTOMATIC_SELECTION=False but no manual override was provided for protocol '{protocol_name}'.")

        criterion_used = criterion_effective_global
        chosen_row = pick_best_row_with_fallback(df, criterion_used)
        chosen_k = int(chosen_row["K"])

        if criterion_used in ("AIC", "BIC"):
            metric_col_used = criterion_used
        else:
            s = pd.Series(df["val_logL"].astype(float))
            metric_col_used = "val_logL" if int(s.notna().sum()) > 0 else "train_logL"

        print(f"[EVAL] Auto selection: chosen K={chosen_k} using '{metric_col_used}' (criterion requested: '{crit_requested}').")

    # Resolve 'best_model_path' & load the trained model (for immediate downstream use):
    chosen_model_path = chosen_row.get("best_model_path", None)
    if chosen_model_path is None or str(chosen_model_path).strip() == "":
        raise RuntimeError(
            f"[EVAL ERROR] Missing 'best_model_path' in comparison table for protocol='{protocol_name}', K={chosen_k}.")
    chosen_model_path = Path(str(chosen_model_path))
    if not chosen_model_path.exists():
        raise RuntimeError(
            f"[EVAL ERROR] best_model_path does not exist for protocol='{protocol_name}', K={chosen_k}:\n  {chosen_model_path}")

    chosen_model_obj = joblib.load(chosen_model_path)

    # Build chosen summary payload (written to 'chosen_model.json'):
    chosen_summary = {
        "protocol": protocol_name,
        "chosen_K": int(chosen_k),
        "chosen_by_manual_override": bool(chosen_by_manual),
        "criterion_requested": crit_requested if AUTOMATIC_SELECTION else None,
        "criterion_used": criterion_used,
        "metric_column_used": metric_col_used,
        "chosen_row": chosen_row,
        "chosen_model_path": str(chosen_model_path),

        # Key patch -- explicitly record PCA needed to USE the chosen interim model (if any):
        "pca_enabled_for_protocol": bool(pca_enabled),
        "pca_fit_on_X_train_path": pca_fit_on_train_path,

        "refit_on_all_data": bool(REFIT_ON_ALL_DATA),
        "refit_model_path": None,
        "refit_pca_path": None,
        "notes": []}

    # Optional refit on 'X_all' (if toggled), & then save final model:
    if REFIT_ON_ALL_DATA:
        cov_type = str(proto_cfg["covariance_type"]).strip().lower()

        pca_n_components = pca_cfg.get("n_components", None)
        pca_whitening = bool(pca_cfg.get("whitening", False))

        X_all_eff = X_all
        pca_refit_obj = None

        if pca_enabled:
            print("[EVAL] REFIT_ON_ALL_DATA=True: refitting PCA on X_all for final model.")
            seed_is_int = isinstance(RANDOM_SEED, (int, np.integer))
            base_seed = int(RANDOM_SEED) if seed_is_int else None

            pca_refit_obj = PCA(
                n_components=pca_n_components,
                whiten=pca_whitening,
                random_state=base_seed)
            X_all_eff = pca_refit_obj.fit_transform(X_all)

            pca_refit_path = protocol_dir / "pca_refit_on_X_all.joblib"
            joblib.dump(pca_refit_obj, pca_refit_path)
            chosen_summary["refit_pca_path"] = str(pca_refit_path)

        # Fit final model on ALL data:
        seed_is_int = isinstance(RANDOM_SEED, (int, np.integer))
        final_random_state = int(RANDOM_SEED) if seed_is_int else None

        final_model = GaussianHMM(
            n_components=int(chosen_k),
            covariance_type=cov_type,
            n_iter=int(proto_cfg.get("max_iterations", 500)),
            tol=float(proto_cfg.get("convergence_tolerance", 1e-4)),
            random_state=final_random_state,
            verbose=False)

        print("[EVAL] Fitting final model on X_all...")
        final_model.fit(X_all_eff, lengths_all)

        final_model_path = protocol_dir / f"final-model_k{int(chosen_k)}.joblib"
        joblib.dump(final_model, final_model_path)
        chosen_summary["refit_model_path"] = str(final_model_path)

        print(f"[EVAL] Saved final refit model: {final_model_path}")

    # Write 'chosen_model.json':
    chosen_json_path = protocol_dir / "chosen_model.json"
    with open(chosen_json_path, "w") as f:
        json.dump(chosen_summary, f, indent=2)
    print(f"[EVAL] Wrote: {chosen_json_path}")

    # Record in-memory outputs:
    CHOSEN_MODELS_BY_PROTOCOL[protocol_name] = {
        "protocol": protocol_name,
        "chosen_K": int(chosen_k),
        "chosen_row": chosen_row,
        "chosen_model_path": str(chosen_model_path),
        "model": chosen_model_obj,

        # Key patch -- carry PCA path in-memory too!:
        "pca_fit_on_X_train_path": pca_fit_on_train_path,
        "refit_pca_path": chosen_summary.get("refit_pca_path", None),

        "chosen_summary": chosen_summary}

    # Winner row for compact review table:
    row_for_table = dict(chosen_row)
    row_for_table["protocol"] = protocol_name
    row_for_table["chosen_by_manual_override"] = bool(chosen_by_manual)
    row_for_table["criterion_effective"] = criterion_used if criterion_used is not None else "MANUAL"
    # Optional: expose PCA status in the winners table:
    row_for_table["pca_enabled_for_protocol"] = bool(pca_enabled)
    winner_rows.append(row_for_table)

# Build 'winners' table:
winners_df = pd.DataFrame(winner_rows)

preferred = [
    "protocol",
    "K",
    "criterion_effective",
    "train_logL",
    "AIC",
    "BIC",
    "val_logL",
    "chosen_by_manual_override",
    "pca_enabled_for_protocol",
    "best_model_path"]
cols = [c for c in preferred if c in winners_df.columns] + [c for c in winners_df.columns if c not in preferred]
winners_df = winners_df[cols].sort_values(["protocol"]).reset_index(drop=True)

print("\n[EVAL] Selected best model per protocol (within-protocol only):")
display(winners_df)

print("\n[EVAL] Winners are now available in CHOSEN_MODELS_BY_PROTOCOL (one entry per protocol).")

----------